# 5 · Server-side gating · the `bm:servable` bitmap + Lua script

`precomputed_segment` still has to fetch every candidate's
`campaign_state:` to check pacing and budget. That's a per-candidate
HMGET burst even after the precompute narrows things down.

The next move: encode "is this campaign currently servable?" as a single
**bitmap** keyed by campaign-bit-index, populated by the same writer that
updates pacing state. Then run the candidate-vs-bitmap join *server-side*
in Redis with a tiny Lua script. The bid path comes back as a list of
gated candidate IDs in **one** round trip — no `campaign_state:` fetch
required.


In [1]:
from notebooks._demo_setup import connect_redis, StepTimer
client = connect_redis()

connected to redis://localhost:6381/0
  users=4000  campaigns=2500  precompute_version=v17_2500_12


## What's in `bm:servable`

A single global Redis BITMAP. Each campaign has a bit position derived
from its ID (`c00042` → bit 42). The bit is set iff the campaign is
both `pacing_status == 'active'` **and** `spent_today_usd <
daily_budget_usd`. Updated alongside any pacing-state write.


In [2]:
bitcount = client.bitcount('bm:servable')
total_campaigns = int(client.get('meta:campaign_count'))
print(f'bm:servable bitcount = {bitcount}')
print(f'total campaigns      = {total_campaigns}')
print(f'fraction servable    = {bitcount/total_campaigns:.1%}')

# Sample: is c00042 servable right now?
bit_index = 42
print(f'\nGETBIT bm:servable {bit_index} = {client.getbit("bm:servable", bit_index)}')

bm:servable bitcount = 2158
total campaigns      = 2500
fraction servable    = 86.3%

GETBIT bm:servable 42 = 1


## The Lua script

This is the actual script registered by `app/repository.py`. It's small
on purpose — Redis Lua is single-threaded against the keyspace, so the
script must be O(N) in the candidate list and nothing fancier.


In [3]:
import inspect
from app.repository import RedisRepository

# Re-extract the script source from the class so the notebook stays in sync
# with whatever ships in the repo.
source = inspect.getsource(RedisRepository.__init__)
script_start = source.index('register_script(')
script_end = source.index('"""', source.index('"""', script_start) + 3)
print(source[script_start:script_end + 3])

register_script(
            """
            local payload = redis.call('GET', KEYS[1])
            if not payload then
                return {}
            end
            local candidate_ids = cjson.decode(payload)
            local max_results = tonumber(ARGV[1])
            local gated = {}
            for _, campaign_id in ipairs(candidate_ids) do
                local bit_index = tonumber(string.sub(campaign_id, 2))
                if redis.call('GETBIT', KEYS[2], bit_index) == 1 then
                    table.insert(gated, campaign_id)
                    if #gated >= max_results then
                        break
                    end
                end
            end
            return gated
            """


Walking through the script:

1. `GET KEYS[1]` — load the precomputed `aud:{maid_id}` JSON list.
2. Parse it.
3. For each candidate ID, `GETBIT KEYS[2] bit_index` against `bm:servable`.
4. Keep only the ones where the bit is set, capped at `ARGV[1]` results.

Everything happens inside Redis on a single shard with the candidate-list
key. From the bid engine's perspective: one round trip, returns the gated
list.


## Running the gate end-to-end

This is the `hybrid_bitmap_gating` mode.


In [4]:
from app.candidate import filter_campaigns_for_user
from app.models import ScoringProfile, Campaign
from app.ranking import rerank_campaigns

# Use the prototype's repository class — it already has the script registered.
from app.repository import RedisRepository
repo = RedisRepository('redis://localhost:6381/0')

IDENTITY_TOKEN = 'id_00042_01'
timer = StepTimer()

with timer.step('identity_resolution'):
    maid_id, _ = repo.resolve_identity(IDENTITY_TOKEN)

with timer.step('hot_profile_fetch'):
    scoring, _ = repo.fetch_scoring_profile(maid_id)

with timer.step('bitmap_gated_candidate_fetch'):
    # One round trip — Lua does the AUD-load + per-candidate GETBIT inline.
    gated_ids, _ = repo.fetch_bitmap_gated_user_candidates(maid_id, limit=200)

with timer.step('campaign_fetch_pipelined'):
    campaigns, _ = repo.fetch_campaigns(gated_ids)

with timer.step('fcap_fetch'):
    fcap_counts, _ = repo.fetch_frequency_caps(maid_id, gated_ids)

with timer.step('frequency_only_filter'):
    eligible = [
        c for c in campaigns
        if fcap_counts.get(c.campaign_id, 0) < c.frequency_cap
    ]

with timer.step('rerank'):
    top_5 = rerank_campaigns(scoring, eligible, top_k=5)

print(f'maid_id            = {maid_id}')
print(f'aud candidates     = (input to bitmap gate)')
print(f'bitmap-gated       = {len(gated_ids)}  (returned in 1 round trip)')
print(f'eligible after fcap = {len(eligible)}')
print(f'top 5: {[(r.campaign_id, round(r.score, 4)) for r in top_5]}')
print()
print(timer.summary())

maid_id            = maid_00042
aud candidates     = (input to bitmap gate)
bitmap-gated       = 21  (returned in 1 round trip)
eligible after fcap = 17
top 5: [('c01011', 5.8237), ('c00848', 4.8107), ('c01551', 4.4065), ('c01222', 4.2812), ('c02229', 4.1223)]

             identity_resolution    3.595 ms
               hot_profile_fetch    0.371 ms
    bitmap_gated_candidate_fetch    6.487 ms
        campaign_fetch_pipelined    3.182 ms
                      fcap_fetch    0.510 ms
           frequency_only_filter    0.006 ms
                          rerank    0.127 ms
--------------------------------------------
                           TOTAL   14.278 ms


## What the bitmap saves us

Compared to `precomputed_segment`:

- the per-candidate `campaign_state:` fanout disappears (fewer round trips
  *and* less app-side state-merge logic),
- the candidate count drops further because the bitmap pre-filters out
  campaigns that have gone unservable since the precompute was built.

On a tuned VM, `hybrid_bitmap_gating` lands at `~1.9 ms` p50 — the fastest
mode that doesn't evaluate the per-campaign taxonomy filter.

But it doesn't evaluate the per-campaign taxonomy filter, which is the
question the customer's PDF actually asks about. That's notebook 6.
